# TP7 — Préparer du texte : NLTK, spaCy et l'effet sur un modèle
**ÉNONCÉ** · Durée estimée : **une demi-journée**

## 🎯 Ce que vous allez apprendre
- Pourquoi un modèle ne sait pas lire du texte brut, et ce que veut dire **pré-traiter**
- Découper en mots, retirer les **mots vides**, ramener les mots à une forme de base : **racine (NLTK)** contre **lemme (spaCy)**
- **Extraire des entités** (personnes, lieux, organisations) et repérer les **ambiguïtés**
- Transformer le texte en nombres (**TF-IDF**) et **mesurer honnêtement** ce que le prétraitement apporte

## 🧰 Boîte à outils
Noms de fonctions et de classes utiles. **Aucun exemple d'appel n'est donné** : cherchez dans la documentation (scikit-learn, Keras, NumPy, matplotlib) comment les utiliser.

`RegexpTokenizer`  ·  `tokenize`  ·  `PorterStemmer`  ·  `stem`  ·  `nlp.Defaults.stop_words`  ·  `token.lemma_`  ·  `token.is_alpha`  ·  `token.is_stop`  ·  `doc.ents`  ·  `ent.label_`  ·  `spacy.explain`  ·  `nlp.pipe`  ·  `TfidfVectorizer`  ·  `LogisticRegression`  ·  `Counter`

## 📘 Comment travailler
- Les cellules **déjà écrites** sont à lire et à exécuter.
- Les cellules **🎯 Étape** sont à **écrire vous-même**, à partir de la consigne.
- Après chaque étape, une cellule **✅ Contrôle** vérifie votre résultat. Respectez les **noms de variables** demandés.
- Les questions **✍️** demandent une réponse écrite : c'est là que vous montrez que vous avez compris.
- Bloqué ? Relisez l'indice, puis cherchez dans la documentation. Un message d'erreur se lit : la dernière ligne dit souvent ce qui ne va pas.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import spacy
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
nlp = spacy.load("en_core_web_sm", disable=["parser"])

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Les données : 2 400 résumés de films, horreur ou comédie ?
Un **résumé de film** est du **texte brut** : ce qu'on appelle une donnée **non structurée** (pas de colonnes, pas de nombres). Notre but : deviner si le film est une **horreur (1)** ou une **comédie (0)** à partir de son seul résumé. Le fichier contient 1 200 textes de chaque sorte, donc une classification **équilibrée**.

Si vous n'avez pas encore installé les outils : `pip install spacy nltk` puis `python -m spacy download en_core_web_sm`.

In [ ]:
df = pd.read_csv("textes_films_horreur_comedie.csv")
print(df.shape, "| horreur (1) :", int(df["cible"].sum()), "| comédie (0) :", int((df["cible"] == 0).sum()))
for i in (0, 1, 2):
    print("•", "HORREUR" if df["cible"][i] else "COMÉDIE", "|", df["resume"][i][:220], "...\n")

## 🎯 Étape 2 — Étape 1 du prétraitement : découper en mots (tokenisation)
Un modèle ne lit pas des phrases : il compte des **mots**. La première opération consiste à **découper** le texte en unités, appelées **tokens**. On passe aussi en **minuscules**, pour que « Killer » et « killer » soient le même mot : c'est de la **normalisation**.

**À faire :** Prenez le **premier résumé** de `df` (`texte`). Avec un **`RegexpTokenizer` de NLTK** qui ne garde que les suites de lettres (`[a-zA-Z]+`), découpez-le en mots, **en minuscules**. Stockez la liste dans `mots` et affichez le nombre de mots et les 15 premiers.

> 💡 **Indice** : `RegexpTokenizer(r"[a-zA-Z]+")` puis la méthode `.tokenize(...)`. Pour les minuscules : `.lower()` sur le texte.

In [ ]:
texte = df["resume"].iloc[0]
# TODO : créer decoupeur et mots


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle(len(mots) > 20 and all(m == m.lower() for m in mots), 'Texte découpé en %d mots, tous en minuscules' % len(mots), 'Attendu : une liste de mots en minuscules')

## 🎯 Étape 3 — Étape 2 : retirer les mots vides (stop words)
Dans un texte, beaucoup de mots (« the », « of », « and », « he »…) sont partout, dans les horreurs comme dans les comédies : ils n'aident pas à les distinguer. On les appelle des **mots vides**. Les retirer **réduit le bruit**.

**À faire :** `spaCy` fournit une liste de mots vides anglais : `nlp.Defaults.stop_words`. Stockez-la dans `mots_vides`, puis gardez dans `mots_utiles` uniquement les mots de `mots` qui n'y figurent pas. Affichez le nombre de mots avant et après.

> 💡 **Indice** : Une liste en compréhension : `[m for m in mots if m not in mots_vides]`.

In [ ]:
# TODO : mots_vides et mots_utiles


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle(len(mots_utiles) < len(mots) and 'the' not in mots_utiles, 'Mots vides retirés : %d -> %d mots' % (len(mots), len(mots_utiles)), 'Attendu : moins de mots qu\'avant, et plus de « the »')

## 🎯 Étape 4 — Étape 3 : ramener les mots à une racine avec NLTK (stemming)
« kill », « killed », « killer » parlent de la même chose. Le **stemming** coupe mécaniquement les fins de mots pour obtenir une **racine** (qui n'est pas toujours un vrai mot). C'est rapide, mais « bête » : il ne connaît pas la grammaire.

**À faire :** Avec le **`PorterStemmer` de NLTK**, calculez la racine de chaque mot de la phrase ci-dessous. Stockez la liste dans `racines` et affichez les couples (mot, racine).

> 💡 **Indice** : `PorterStemmer()` puis `.stem(mot)`.

In [ ]:
phrase = "The killers ran away while he was studying better methods"
mots_phrase = phrase.lower().split()
# TODO : racineur et racines


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle(racines[mots_phrase.index('studying')] == 'studi' and racines[mots_phrase.index('ran')] == 'ran', 'Racines calculées (remarquez « studi » et « ran »)', 'Attendu : studying -> studi, et ran reste ran')

### ✍️ Question
**Regardez « studying → studi » et « ran → ran ». Qu'est-ce que le stemming sait faire, et qu'est-ce qu'il ne sait pas faire ?**

*Votre réponse (2 à 4 lignes) :*

> 

## 🎯 Étape 5 — Étape 4 : ramener les mots à leur forme de base avec spaCy (lemmatisation)
La **lemmatisation** fait la même chose en **comprenant la grammaire** : le **lemme** est le mot du dictionnaire (« ran » → « run », « was » → « be »). C'est plus lent mais plus juste.

**À faire :** Passez la même `phrase` dans spaCy (`doc = nlp(phrase)`). Construisez un tableau pandas **`tableau`** de 10 lignes avec 3 colonnes : `mot`, `racine NLTK` (réutilisez `racines`) et `lemme spaCy`. Affichez-le.

> 💡 **Indice** : Pour chaque `token` de `doc` : `token.text` et `token.lemma_`.

In [ ]:
# TODO : doc et tableau


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle(tableau.shape == (10, 3) and tableau.loc[tableau['mot'] == 'ran', 'lemme spaCy'].iloc[0] == 'run' and tableau.loc[tableau['mot'] == 'was', 'lemme spaCy'].iloc[0] == 'be', 'Tableau correct : ran -> run, was -> be', 'Attendu 10 lignes, avec ran -> run et was -> be')

### ✍️ Question
**Comparez les colonnes « racine NLTK » et « lemme spaCy ». Dans quel cas préféreriez-vous l'une ou l'autre ?**

*Votre réponse (2 à 4 lignes) :*

> 

## 🎯 Étape 6 — Étape 5 : extraire les entités nommées
Une **entité nommée** est un nom propre de personne, de lieu, d'organisation, une date, un nombre… spaCy les repère automatiquement. Extraire les entités permet de savoir **de qui, d'où, de quand** parle un texte, et de créer des variables utiles.

**À faire :** 1) Passez le **premier résumé** dans spaCy et affichez ses entités (texte → type, avec la signification du type). 2) Passez les **300 premiers résumés** dans `nlp.pipe(...)` et comptez les **types d'entités** dans `types` (un `Counter`).

> 💡 **Indice** : `for ent in doc.ents:` donne `ent.text` et `ent.label_`. `spacy.explain("GPE")` donne la signification. `nlp.pipe(liste_de_textes, batch_size=64)` traite beaucoup de textes plus vite qu'un par un.

In [ ]:
# TODO : afficher les entités du 1er résumé, puis calculer docs300 et types


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle('PERSON' in types and sum(types.values()) > 300, 'Entités extraites : %d dans 300 résumés' % sum(types.values()), 'Attendu des entités de type PERSON et plus de 300 au total')

## 🎯 Étape 7 — Étape 6 : chasser les ambiguïtés
Une extraction automatique n'est **pas parfaite** : un même nom peut recevoir **plusieurs étiquettes** selon le contexte (un prénom pris pour une organisation, un lieu pour une personne…). Le référentiel parle de **réduction des ambiguïtés** : il faut d'abord **les repérer**.

**À faire :** À partir de `docs300`, construisez le dictionnaire `etiquettes` (clé = texte de l'entité, valeur = **ensemble** des types qu'elle a reçus), puis `ambigus` : celles qui ont **au moins 2 types différents**. Affichez le nombre d'entités ambiguës et **10 exemples**.

> 💡 **Indice** : `etiquettes.setdefault(ent.text, set()).add(ent.label_)`.

In [ ]:
etiquettes = {}
# TODO : remplir etiquettes, puis calculer ambigus


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle(len(ambigus) > 0, '%d entités ambiguës repérées' % len(ambigus), 'Attendu au moins une entité avec plusieurs types')

### ✍️ Question
**Quelles ambiguïtés avez-vous trouvées ? Que feriez-vous pour fiabiliser ces extractions avant d'entraîner un modèle ?**

*Votre réponse (2 à 4 lignes) :*

> 

## 🎯 Étape 8 — Étape 7 : le pipeline complet
On assemble tout : un texte brut entre, un **texte propre** sort, en minuscules, **sans ponctuation, sans mots vides, avec des lemmes**.

**À faire :** Écrivez `nettoyer_doc(doc)` qui reçoit un document spaCy et renvoie une chaîne : les **lemmes en minuscules** des tokens qui sont **alphabétiques** (`is_alpha`) et **pas des mots vides** (`is_stop`), séparés par des espaces. Appliquez-la à **tous** les résumés dans une colonne `df["propre"]` (⏱️ environ 20 secondes).

> 💡 **Indice** : `nlp.pipe(df["resume"], batch_size=128)` donne les documents un par un. Les attributs : `token.lemma_`, `token.is_alpha`, `token.is_stop`.

In [ ]:
def nettoyer_doc(doc):
    # TODO
    pass

# TODO : remplir df["propre"]


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle('propre' in df.columns and df['propre'].notna().all() and ',' not in df['propre'].iloc[0] and df['propre'].str.len().min() > 0, 'Colonne propre créée pour les %d textes' % len(df), 'Vérifiez nettoyer_doc : pas de ponctuation, aucun texte vide')

## Étape 9 — Deux autres versions du texte, pour comparer
On prépare aussi une version « **minuscules** » et une version « **racines NLTK** » (avec les outils des étapes 1 à 3), pour comparer les traitements à l'étape 9.

In [ ]:
def racines_texte(texte):
    return " ".join(racineur.stem(m) for m in decoupeur.tokenize(texte.lower()) if m not in mots_vides)

df["minuscules"] = df["resume"].str.lower()
df["racines"] = df["resume"].apply(racines_texte)
print(df[["resume", "minuscules", "racines", "propre"]].iloc[0].apply(lambda s: s[:90]).to_string())

## 🎯 Étape 10 — Étape 8 : transformer le texte en nombres, et entraîner
Pour qu'un modèle lise un texte, on le transforme en **nombres** : le **TF-IDF** donne à chaque mot un poids élevé s'il est **fréquent dans ce texte** et **rare dans les autres** (« killer » aide ; « the » n'aide pas). Le **vocabulaire** (la liste des mots connus) **s'apprend sur l'apprentissage seul**, sinon le test « souffle » des informations : c'est une **fuite de données**.

**À faire :** 1) Découpez `df` en `train` / `test` : **25 % de test**, **stratifié sur `cible`**, `random_state=42`. 2) Pour chaque colonne **`resume`** (texte brut) et **`propre`** (texte nettoyé) : apprenez un `TfidfVectorizer` **sur `train` seulement**, entraînez une `LogisticRegression(max_iter=2000)`, puis calculez sur `test` la **précision**, le **F1** et la **taille du vocabulaire**. Rangez dans un dictionnaire `resultats` (clés `"resume"` et `"propre"`) et affichez-le en tableau.

> 💡 **Indice** : `vec.fit_transform(train[col])` sur l'apprentissage, `vec.transform(test[col])` sur le test. `X.shape[1]` = nombre de mots du vocabulaire.

In [ ]:
# TODO : train / test, puis les deux entraînements, puis resultats


In [ ]:
# ✅ Contrôle (ne pas modifier)
controle(len(train) == 1800 and 0.7 < resultats['resume']['précision'] < 0.95 and resultats['propre']['mots du vocabulaire'] < resultats['resume']['mots du vocabulaire'], 'Comparaison faite : vocabulaire réduit à %d mots' % resultats['propre']['mots du vocabulaire'], 'Attendu : 1800 textes d\'apprentissage, précisions entre 0.7 et 0.95, vocabulaire réduit après nettoyage')

## Étape 11 — Étape 9 : le prétraitement améliore-t-il vraiment ? Mesurons sur 5 découpages
Un seul découpage ne prouve rien : le hasard du découpage fait varier le résultat. On **répète 5 fois** avec des découpages différents et on regarde la **moyenne** et l'**écart-type** (± : la dispersion). On le fait aussi avec **200 textes seulement**, pour voir ce qui change quand les données sont rares.

⏱️ Quelques dizaines de secondes.

In [ ]:
def evaluer(colonne, taille=None, **options):
    scores, voc = [], []
    for graine in range(5):
        tr, te = train_test_split(df, test_size=0.25, stratify=df["cible"], random_state=graine)
        if taille:
            tr = tr.sample(taille, random_state=graine)
        vec = TfidfVectorizer(**options); X = vec.fit_transform(tr[colonne])
        m = LogisticRegression(max_iter=2000).fit(X, tr["cible"])
        scores.append(np.mean(m.predict(vec.transform(te[colonne])) == te["cible"])); voc.append(X.shape[1])
    return np.mean(scores), np.std(scores), int(np.mean(voc))

configs = [("texte brut (sans minuscules)", "resume", dict(lowercase=False)), ("minuscules", "minuscules", {}),
           ("minuscules + mots vides retirés", "minuscules", dict(stop_words="english")),
           ("racines NLTK + mots vides retirés", "racines", {}), ("lemmes spaCy + mots vides retirés", "propre", {})]
lignes = []
for nom, col, opt in configs:
    m1, s1, v1 = evaluer(col, **opt); m2, s2, _ = evaluer(col, taille=200, **opt)
    lignes.append({"traitement": nom, "précision (1800 textes)": f"{m1:.3f} ± {s1:.3f}", "précision (200 textes)": f"{m2:.3f} ± {s2:.3f}", "mots du vocabulaire": v1})
bilan = pd.DataFrame(lignes)
bilan

### ✍️ Question
**Quelle étape de prétraitement améliore le plus la précision ? Les racines et les lemmes apportent-ils autre chose ? Que se passe-t-il avec 200 textes seulement ?**

*Votre réponse (2 à 4 lignes) :*

> 

## ✅ Ce qu'il faut retenir
- Un modèle ne lit pas du texte : on **découpe** (tokens), on **normalise** (minuscules), on **retire les mots vides**, on ramène les mots à une **forme de base** (racine NLTK ou lemme spaCy), puis on **vectorise** (TF-IDF).
- **Racine** (rapide, approximative, « studi ») contre **lemme** (grammaticalement juste, « study », plus lent).
- Les **entités nommées** donnent le « qui, où, quand », mais l'extraction est **imparfaite** : il faut **chercher les ambiguïtés**.
- Le **vocabulaire** s'apprend **sur l'apprentissage seul**.
- **Mesurer** : un prétraitement améliore parfois nettement (mots vides), parfois seulement la taille du modèle (racines, lemmes), et avec peu de données le hasard masque tout.

## 🧭 Lien avec le référentiel (compétence C.14.3)
| Ce que demande le référentiel | Où dans ce TP |
|---|---|
| Outils NLP comme **SpaCy et NLTK** | NLTK : étapes 1 et 3 · spaCy : étapes 2, 4, 5, 6, 7 |
| **Extraction précise des entités** | Étapes 5 et 6 |
| **Normalisation du texte** | Étapes 1 à 4 et 7 |
| **Réduction des ambiguïtés** | Étape 6 (repérage), étape 3 à 4 (formes d'un même mot) |
| **Optimiser les performances** des modèles | Étapes 8 et 9 |